# 04 -- Celery et workers

## Objectifs pedagogiques

A la fin de ce notebook, vous saurez :

- comprendre l'architecture producteur/broker/worker ;
- configurer une application Celery ;
- definir des taches avec `@app.task` ;
- appeler des taches synchronement et asynchronement ;
- recuperer les resultats avec un backend ;
- chaines de taches, groupes, chords ;
- planifier des taches avec Celery Beat ;
- gerer les erreurs et retries ;
- monitorer avec Flower.

## Prerequis -- ce que vous devez maitriser

- Python intermediaire, decorateurs ;
- notions de messaging et file d'attente ;
- installation d'un broker (Redis ou RabbitMQ) en local.

Ce que ce notebook **ne couvrira pas** :

- le deploiement en production (k8s, systemd) ;
- l'async natif (`asyncio`) ;
- les alternatives (Huey, RQ, Dramatiq) en detail.

## Plan

1. Architecture Celery
2. Installation et broker
3. Premiere tache
4. Appel synchrone vs asynchrone
5. Backend de resultats
6. Arguments et serialisation
7. Retries et gestion d'erreurs
8. Chaines de taches
9. Groupes et chords
10. Taches periodiques (Beat)
11. Routing et queues
12. Monitoring avec Flower
13. Alternatives a Celery
14. Synthese
15. Exercices

---

## 1. Architecture Celery

```
[Producteur] --publie--> [Broker] --consomme--> [Worker 1]
                                              |--> [Worker 2]
                                              |--> [Worker N]
                                          (resultat)
                                              v
                                          [Backend]
```

- **Broker** : stocke les messages (Redis, RabbitMQ)
- **Worker** : process qui execute les taches
- **Backend** : stocke les resultats (Redis, DB, S3)

---

## 2. Installation et broker

In [ ]:
# pip install 'celery[redis]'
# Broker Redis : docker run -p 6379:6379 redis
import celery
celery.__version__

---

## 3. Premiere tache

In [ ]:
from celery import Celery

app = Celery(
    'demo',
    broker='redis://localhost:6379/0',
    backend='redis://localhost:6379/0',
)

@app.task
def add(x: int, y: int) -> int:
    return x + y

print('tache enregistree :', add.name)

Dans un fichier `tasks.py`, on lance un worker avec :

```bash
celery -A tasks worker --loglevel=info
```

En notebook, on appelle les taches directement pour la demo.

---

## 4. Appel synchrone vs asynchrone

In [ ]:
# Sync (pas de broker)
result = add(2, 3)
print('sync :', result)

In [ ]:
# Async : envoie au broker, retourne un AsyncResult
# async_result = add.delay(2, 3)
# print(async_result.id, async_result.status)
print('Pour async, worker doit tourner')

In [ ]:
# Avec apply_async : plus de controle
# add.apply_async(args=(2, 3), countdown=10, expires=60)
print('apply_async: countdown, eta, expires, queue...')

---

## 5. Backend de resultats

In [ ]:
# async_result = add.delay(10, 20)
# async_result.get(timeout=5)  # bloque jusqu'au resultat
# async_result.status   # PENDING / STARTED / SUCCESS / FAILURE
print('Backends : redis, db, s3, rpc, ignore_result')

---

## 6. Arguments et serialisation

In [ ]:
@app.task
def process_user(user_data: dict) -> dict:
    return {'processed': True, 'user': user_data['name']}

print(process_user({'name': 'Alice', 'age': 30}))

### Serializers supportes

- `json` (defaut, securise)
- `pickle` (non securise, evite en production)
- `yaml`, `msgpack`

On configure avec `app.conf.task_serializer = 'json'`.

---

## 7. Retries et gestion d'erreurs

In [ ]:
@app.task(bind=True, max_retries=3, default_retry_delay=5)
def fetch_url(self, url: str) -> str:
    try:
        # return requests.get(url).text
        return f'fetched {url}'
    except Exception as exc:
        raise self.retry(exc=exc)

print('retry configure')

In [ ]:
# Autoretry avec classe d'exception
@app.task(autoretry_for=(ConnectionError,), retry_backoff=True, max_retries=5)
def flaky() -> str:
    return 'ok'

print('autoretry + backoff exponentiel')

---

## 8. Chaines de taches

In [ ]:
from celery import chain

@app.task
def double(x: int) -> int:
    return x * 2

@app.task
def increment(x: int) -> int:
    return x + 1

# Chaine : double(5) -> increment(10) -> double(11)
# workflow = chain(double.s(5), increment.s(), double.s())
# workflow().get()
print('Chaine creee (requiert worker)')

---

## 9. Groupes et chords

In [ ]:
from celery import group, chord

# Groupe : execute en parallele, collecte tous les resultats
# g = group(double.s(i) for i in range(5))
# g().get()  # [0, 2, 4, 6, 8]
print('Group = execution parallele')

In [ ]:
# Chord : groupe + callback
# c = chord(group(double.s(i) for i in range(5)), sum_all.s())
# c().get()
@app.task
def sum_all(results: list[int]) -> int:
    return sum(results)

print('Chord = group + callback d\'agregation')

---

## 10. Taches periodiques (Beat)

In [ ]:
from celery.schedules import crontab

app.conf.beat_schedule = {
    'ping-every-30s': {
        'task': 'demo.add',
        'schedule': 30.0,  # secondes
        'args': (1, 2),
    },
    'report-hourly': {
        'task': 'demo.add',
        'schedule': crontab(minute=0),  # toutes les heures
        'args': (0, 0),
    },
}
print('Beat configure :', list(app.conf.beat_schedule.keys()))

Pour lancer : `celery -A tasks beat --loglevel=info` (en plus des workers).

---

## 11. Routing et queues

In [ ]:
app.conf.task_routes = {
    'demo.send_email': {'queue': 'emails'},
    'demo.heavy_compute': {'queue': 'compute'},
}
print('Routes :', app.conf.task_routes)

Lancer un worker dedie :

```bash
celery -A tasks worker -Q emails --concurrency=4
celery -A tasks worker -Q compute --concurrency=2
```

---

## 12. Monitoring avec Flower

**Flower** est un dashboard web pour Celery.

```bash
pip install flower
celery -A tasks flower --port=5555
```

Ouvre une UI sur http://localhost:5555 avec :

- taches en cours, reussies, echouees
- temps d'execution, graphiques
- workers actifs
- possibilite de revoker ou retry

---

## 13. Alternatives a Celery

| Outil | Points forts |
|---|---|
| **RQ** | simple, base redis, moins de features |
| **Dramatiq** | moderne, simple, priorites natives |
| **Huey** | tres leger, sqlite ou redis |
| **APScheduler** | scheduler pur Python, pas de broker |
| **ARQ** | async-first (asyncio) |
| **Taskiq** | async + middleware |

---

## Synthese

| Element | Celery |
|---|---|
| App | `Celery('name', broker=..., backend=...)` |
| Tache | `@app.task def f(...): ...` |
| Async | `f.delay(...)` ou `f.apply_async(args=, ...)` |
| Resultat | `async_result.get(timeout=5)` |
| Retry | `self.retry(exc=exc)` ou `autoretry_for=` |
| Chaine | `chain(task.s(...), ...)` |
| Groupe | `group(task.s(i) for i in range(n))` |
| Chord | `chord(group(...), callback.s())` |
| Beat | `app.conf.beat_schedule = {...}` |
| Queue | `app.conf.task_routes = {...}` |

### Regles a retenir

- **Broker obligatoire** : Redis (simple) ou RabbitMQ (robuste).
- **`delay` = `apply_async` simplifie**.
- **Taches idempotentes** : elles peuvent etre retentees.
- **JSON serializer par defaut**, jamais pickle en prod.
- **Beat** pour les taches planifiees, pas pour les taches immediates.

---

## Exercices

### Exercice 1 (facile) -- Tache qui fait un gros calcul

Definissez une tache `compute_fibonacci(n)` qui renvoie le n-eme nombre de Fibonacci.

In [ ]:
from celery import Celery
app = Celery('demo', broker='redis://localhost/0', backend='redis://localhost/0')

# TODO

<details>
<summary>Voir la correction</summary>

```python
from celery import Celery
app = Celery('demo', broker='redis://localhost/0', backend='redis://localhost/0')

@app.task
def compute_fibonacci(n: int) -> int:
    if n < 2:
        return n
    a, b = 0, 1
    for _ in range(n - 1):
        a, b = b, a + b
    return b

print(compute_fibonacci(10))
```

</details>

### Exercice 2 (facile) -- Envoi d'email simule

Creez une tache `send_email(to, subject, body)` qui simule l'envoi (print) et retourne un dict `{'sent': True, ...}`.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
@app.task
def send_email(to: str, subject: str, body: str) -> dict:
    print(f'-> envoi a {to}: {subject}')
    return {'sent': True, 'to': to, 'subject': subject}

send_email('alice@x.com', 'Hello', 'Bonjour !')
```

</details>

### Exercice 3 (moyen) -- Retry avec backoff

Creez une tache `fetch_data(url)` qui leve aleatoirement une exception et se retry jusqu'a 3 fois avec backoff exponentiel.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import random

@app.task(
    bind=True,
    autoretry_for=(ConnectionError,),
    retry_backoff=True,
    max_retries=3,
)
def fetch_data(self, url: str) -> dict:
    if random.random() < 0.5:
        raise ConnectionError(f'failed {url}')
    return {'url': url, 'data': 'ok'}
```

</details>

### Exercice 4 (moyen) -- Pipeline avec chain

Chainez 3 taches : `fetch_url` -> `parse` -> `save`.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from celery import chain

@app.task
def fetch_url(url: str) -> str:
    return f'<html>{url}</html>'

@app.task
def parse(html: str) -> dict:
    return {'title': 'Demo', 'raw': html}

@app.task
def save(data: dict) -> int:
    return hash(str(data))

pipeline = chain(fetch_url.s('http://x'), parse.s(), save.s())
# pipeline().get()
print('pipeline construit')
```

</details>

### Exercice 5 (difficile) -- Chord avec agregation

Construisez un chord qui :
1. lance 5 taches `process_chunk(i)` en parallele ;
2. collecte les resultats avec `aggregate(results)`.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from celery import chord, group

@app.task
def process_chunk(i: int) -> int:
    return i * i

@app.task
def aggregate(results: list[int]) -> dict:
    return {'count': len(results), 'total': sum(results)}

workflow = chord(group(process_chunk.s(i) for i in range(5)), aggregate.s())
# workflow().get()
print('chord construit')
```

</details>

---

## Ressources externes

- [Documentation Celery](https://docs.celeryq.dev/)
- [Celery Best Practices](https://denibertovic.com/posts/celery-best-practices/)
- [Flower](https://flower.readthedocs.io/)